In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path("/content/hyperfs")
if not (repo_dir / ".git").is_dir():
    subprocess.run(
        [
            "git", "clone", "--branch", "feature/aastha-month1-data-loader",
            "--single-branch", "https://github.com/aniya-modi/agentfs-track-b.git",
            str(repo_dir),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)

%cd /content/hyperfs
!mkdir -p data/raw data/processed src
!touch src/__init__.py


In [ ]:
!ls -R /content/hyperfs


In [ ]:
import io
import os
import urllib.request
from pathlib import Path

import pandas as pd
import scipy.io

os.chdir("/content/hyperfs")
os.makedirs("data/raw", exist_ok=True)

# Golub_Merge is the checked-in Bioconductor reference export. Its columns
# carry the original Hu6800 probe set IDs; the scikit-feature .mat file has
# only X and Y and cannot restore those IDs.
GOLUB_REFERENCE = Path("data/reference/golub_hu6800.csv.gz")
if not GOLUB_REFERENCE.is_file():
    raise FileNotFoundError(
        f"Missing {GOLUB_REFERENCE}. Clone the project feature branch first."
    )
df_golub = pd.read_csv(GOLUB_REFERENCE, compression="gzip")
if df_golub.shape != (72, 7130):
    raise ValueError(f"Unexpected Golub reference shape: {df_golub.shape}")
if not df_golub.columns.is_unique:
    raise ValueError("Golub reference contains duplicate feature IDs.")
if any(str(column).startswith("GENE_") for column in df_golub.columns):
    raise ValueError("Golub reference must contain original Hu6800 probe IDs.")
df_golub.to_csv("data/raw/golub_raw.csv", index=False)

# Colon.mat contains X and Y but no native feature names. Use deterministic
# source-column identifiers (att_1 ... att_2000); these are not gene/probe IDs.
COLON_MAT_URL = (
    "https://raw.githubusercontent.com/jundongl/scikit-feature/"
    "master/skfeature/data/colon.mat"
)

def fetch_indexed_mat_dataset(url, csv_filename, dataset_name, feature_prefix):
    print(f"Downloading {dataset_name} dataset...")
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request) as response:
        mat = scipy.io.loadmat(io.BytesIO(response.read()))

    X = mat["X"]
    y = mat["Y"].ravel()
    feature_ids = [f"{feature_prefix}_{index + 1}" for index in range(X.shape[1])]
    if len(feature_ids) != X.shape[1] or len(set(feature_ids)) != len(feature_ids):
        raise ValueError(f"Could not create unique IDs for {dataset_name} features.")

    df = pd.DataFrame(X, columns=feature_ids)
    df["target"] = y
    save_path = Path("data/raw") / csv_filename
    df.to_csv(save_path, index=False)
    print(f"Saved {save_path}: {X.shape[0]} samples × {X.shape[1]} indexed features.")
    return df

df_colon = fetch_indexed_mat_dataset(
    COLON_MAT_URL,
    "colon_raw.csv",
    "Alon Colon",
    feature_prefix="att",
)

print("\n--- Dataset ingestion summary ---")
print(f"Golub shape: {df_golub.shape}; first probe IDs: {list(df_golub.columns[:5])}")
print(f"Colon shape: {df_colon.shape}; first source IDs: {list(df_colon.columns[:5])}")


In [ ]:
%%writefile src/data_loader.py
"""
Data Loading and Preprocessing Pipeline for Microarray Gene-Expression Datasets.
Part of the hyperfs pipeline.
"""

import os
import re
from typing import Dict, Optional, Sequence, Tuple

import numpy as np
import pandas as pd


class MicroarrayDataLoader:
    """
    Modular loader and preprocessor for high-dimensional microarray data.
    """
    def __init__(self, file_path: str, label_column: str = 'target'):
        self.file_path = file_path
        self.label_column = label_column
        self.df: Optional[pd.DataFrame] = None
        self.X: Optional[pd.DataFrame] = None
        self.y: Optional[np.ndarray] = None

    def load_raw_data(
        self,
        *,
        feature_ids: Optional[Sequence[str]] = None,
    ) -> Tuple[pd.DataFrame, np.ndarray]:
        """Load features and labels, optionally restoring IDs from a verified source.

        ``feature_ids`` must be in the same order as the feature columns in the
        input CSV. Generic ``GENE_1``-style placeholders are rejected unless
        verified identifiers are supplied explicitly.
        """
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"Dataset file not found at: {self.file_path}")

        self.df = pd.read_csv(self.file_path)

        if self.label_column in self.df.columns:
            self.y = self.df[self.label_column].values
            self.X = self.df.drop(columns=[self.label_column])
        else:
            raise KeyError(f"Label column '{self.label_column}' not found in dataset.")

        if feature_ids is not None:
            ids = [str(feature_id).strip() for feature_id in feature_ids]
            if len(ids) != self.X.shape[1]:
                raise ValueError(
                    f"Expected {self.X.shape[1]} feature IDs, received {len(ids)}."
                )
            if any(not feature_id for feature_id in ids):
                raise ValueError("Feature IDs must not be empty.")
            if len(set(ids)) != len(ids):
                raise ValueError("Feature IDs must be unique.")
            self.X.columns = ids
        elif any(re.fullmatch(r"GENE_\d+", str(column)) for column in self.X.columns):
            raise ValueError(
                "Generic GENE_n feature labels are not identifiers. Pass the verified "
                "feature_ids in source-column order or use a source file with real IDs."
            )

        print(f"Loaded raw dataset: {self.X.shape[0]} samples × {self.X.shape[1]} features.")
        return self.X, self.y

    def apply_log2_transform(self, offset: float = 1.0) -> pd.DataFrame:
        """
        Applies log2 transformation: log2(x + offset) to stabilize expression variance.
        """
        # Ensure values are non-negative before applying log2
        min_val = self.X.values.min()
        if min_val < 0:
            self.X = self.X - min_val

        self.X = np.log2(self.X + offset)
        print("Applied log2 transformation.")
        return self.X

    def handle_missing_values(self) -> pd.DataFrame:
        """Fills missing values with column-wise median."""
        missing_count = self.X.isnull().sum().sum()
        if missing_count > 0:
            self.X = self.X.fillna(self.X.median())
            print(f"Imputed {missing_count} missing values using median strategy.")
        else:
            print("No missing values found.")
        return self.X

    def apply_variance_filter(self, top_k: int = 1000) -> pd.DataFrame:
        """
        Pre-screens features by keeping the top_k features with the highest variance.
        Crucial for mitigating high-dimensionality overhead (p >> n).
        """
        if self.X.shape[1] <= top_k:
            print(f"Feature count ({self.X.shape[1]}) <= top_k ({top_k}). Skipping variance filter.")
            return self.X

        variances = self.X.var(axis=0)
        top_feature_names = variances.nlargest(top_k).index
        self.X = self.X[top_feature_names]
        print(f"Variance Filter Applied: Filtered from {len(variances)} to top {top_k} features.")
        return self.X

    def map_probes(self, mapping_dict: Optional[Dict[str, str]] = None) -> pd.DataFrame:
        """
        Renames probe IDs to official Gene Symbols if mapping dictionary is provided.
        """
        if mapping_dict:
            self.X = self.X.rename(columns=mapping_dict)
            # Aggregate duplicate gene columns by mean expression value
            self.X = self.X.groupby(level=0, axis=1).mean()
            print(f"Probe mapping applied. Retained {self.X.shape[1]} unique mapped genes.")
        else:
            print("No mapping dictionary provided. Keeping original probe IDs.")
        return self.X

    def save_processed(self, output_path: str):
        """Saves the preprocessed X matrix and target y into processed directory."""
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        processed_df = self.X.copy()
        processed_df[self.label_column] = self.y
        processed_df.to_csv(output_path, index=False)
        print(f"Saved preprocessed matrix to: {output_path}")


In [ ]:
import sys
import os

# Ensure src module is discoverable by Python path
if '/content/hyperfs' not in sys.path:
    sys.path.append('/content/hyperfs')

from src.data_loader import MicroarrayDataLoader

print("=== Processing Golub Leukemia Dataset ===")
golub_loader = MicroarrayDataLoader('data/raw/golub_raw.csv', label_column='target')
golub_loader.load_raw_data()
golub_loader.apply_log2_transform()
golub_loader.handle_missing_values()
golub_loader.apply_variance_filter(top_k=1000)
golub_loader.save_processed('data/processed/golub_processed.csv')

print("\n=== Processing Alon Colon Cancer Dataset ===")
colon_loader = MicroarrayDataLoader('data/raw/colon_raw.csv', label_column='target')
colon_loader.load_raw_data()
colon_loader.apply_log2_transform()
colon_loader.handle_missing_values()
colon_loader.apply_variance_filter(top_k=1000)
colon_loader.save_processed('data/processed/colon_processed.csv')

print("\n--- Month 1 Data Processing Complete ---")

In [ ]:
import pandas as pd

# Load and inspect Golub processed dataset
df_golub_proc = pd.read_csv('data/processed/golub_processed.csv')
print("--- Golub Processed Matrix Overview ---")
print(f"Shape: {df_golub_proc.shape}")
print("Sample Probes / Genes:", list(df_golub_proc.columns[:5]))
print("Target Class Distribution:\n", df_golub_proc['target'].value_counts())

print("\n" + "="*40 + "\n")

# Load and inspect Colon processed dataset
df_colon_proc = pd.read_csv('data/processed/colon_processed.csv')
print("--- Colon Processed Matrix Overview ---")
print(f"Shape: {df_colon_proc.shape}")
print("Sample Probes / Genes:", list(df_colon_proc.columns[:5]))
print("Target Class Distribution:\n", df_colon_proc['target'].value_counts())

In [ ]:
# This notebook is cloned from the team repository's feature branch.
# Review and commit your changes locally, then push normally to the same branch.
!git status --short --branch
!git remote -v
# After committing, push with the repository's existing authentication:
# !git push -u origin feature/aastha-month1-data-loader


In [ ]:
!git clone https://github.com/Aastha0305/hyperfs.git
%cd hyperfs

Cloning into 'hyperfs'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 17 (delta 2), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 624.83 KiB | 10.41 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/hyperfs


In [ ]:
%%writefile src/baseline_benchmarks.py
"""
Baseline Feature Selection and Classification Benchmarks.
"""

import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

class BaselineBenchmarkPipeline:
    def __init__(self, data_path: str, target_column: str = 'target', top_k: int = 100):
        self.data_path = data_path
        self.target_column = target_column
        self.top_k = top_k

        if not os.path.exists(self.data_path):
            raise FileNotFoundError(f"Processed data file not found at: {self.data_path}")

        self.df = pd.read_csv(data_path)
        if self.target_column not in self.df.columns:
            self.target_column = self.df.columns[-1]

        self.X = self.df.drop(columns=[self.target_column])

        # Encode target labels to 0 and 1 (fixes XGBoost class expectation error)
        raw_y = self.df[self.target_column]
        self.label_encoder = LabelEncoder()
        self.y = pd.Series(self.label_encoder.fit_transform(raw_y), index=raw_y.index)

    def run_mrmr(self) -> list:
        """Approximates mRMR using Mutual Information relevance ranking."""
        print(f"[*] Running mRMR feature selection (Top {self.top_k}) on {os.path.basename(self.data_path)}...")
        mi = mutual_info_classif(self.X, self.y, random_state=42)
        mi_series = pd.Series(mi, index=self.X.columns)
        return mi_series.nlargest(self.top_k).index.tolist()

    def run_lasso(self) -> list:
        """Runs L1 Regularization (LASSO) via cross-validation for sparse feature selection."""
        print(f"[*] Running LASSO L1 feature selection on {os.path.basename(self.data_path)}...")
        lasso = LassoCV(cv=5, random_state=42, max_iter=5000)
        lasso.fit(self.X, self.y)

        coefficients = pd.Series(lasso.coef_, index=self.X.columns)
        non_zero = coefficients[coefficients != 0]

        if len(non_zero) == 0:
            return coefficients.abs().nlargest(self.top_k).index.tolist()
        return non_zero.abs().nlargest(self.top_k).index.tolist()

    def evaluate_classifiers(self, feature_subset: list, method_name: str) -> pd.DataFrame:
        """Evaluates SVM, Random Forest, Logistic Regression, and XGBoost using 5-Fold Stratified CV."""
        X_sub = self.X[feature_subset]
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        classifiers = {
            "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
            "SVM (RBF Kernel)": SVC(kernel='rbf', probability=True, random_state=42),
            "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
            "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
        }

        results = []
        for name, clf in classifiers.items():
            acc_scores, f1_scores = [], []
            for train_idx, test_idx in skf.split(X_sub, self.y):
                X_train, X_test = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
                y_train, y_test = self.y.iloc[train_idx], self.y.iloc[test_idx]

                clf.fit(X_train, y_train)
                y_pred = clf.predict(X_test)

                acc_scores.append(accuracy_score(y_test, y_pred))
                f1_scores.append(f1_score(y_test, y_pred, average='weighted'))

            results.append({
                "Dataset": os.path.basename(self.data_path).replace('_processed.csv', '').upper(),
                "Method": method_name,
                "Classifier": name,
                "Mean Accuracy": round(np.mean(acc_scores), 4),
                "Std Accuracy": round(np.std(acc_scores), 4),
                "Mean Macro F1": round(np.mean(f1_scores), 4),
                "Selected Features Count": len(feature_subset)
            })

        return pd.DataFrame(results)

Overwriting src/baseline_benchmarks.py


In [ ]:
import os
import pandas as pd
import importlib
import src.baseline_benchmarks
importlib.reload(src.baseline_benchmarks)  # Forces Python to load the new code with LabelEncoder

from src.baseline_benchmarks import BaselineBenchmarkPipeline

os.makedirs('data/results', exist_ok=True)
datasets = ["data/processed/golub_processed.csv", "data/processed/colon_processed.csv"]

all_results = []
for ds_path in datasets:
    if os.path.exists(ds_path):
        print(f"\n--- Processing {os.path.basename(ds_path)} ---")
        pipeline = BaselineBenchmarkPipeline(ds_path, top_k=100)

        # 1. mRMR Evaluation
        mrmr_feats = pipeline.run_mrmr()
        all_results.append(pipeline.evaluate_classifiers(mrmr_feats, method_name="mRMR"))

        # 2. LASSO Evaluation
        lasso_feats = pipeline.run_lasso()
        all_results.append(pipeline.evaluate_classifiers(lasso_feats, method_name="LASSO"))

final_df = pd.concat(all_results, ignore_index=True)
final_df.to_csv("data/results/baseline_benchmark_scores.csv", index=False)

print("\n================ BASELINE BENCHMARK RESULTS ================")
print(final_df.to_string(index=False))
print("\n[+] Benchmarks completed successfully!")


--- Processing golub_processed.csv ---
[*] Running mRMR feature selection (Top 100) on golub_processed.csv...
[*] Running LASSO L1 feature selection on golub_processed.csv...

--- Processing colon_processed.csv ---
[*] Running mRMR feature selection (Top 100) on colon_processed.csv...
[*] Running LASSO L1 feature selection on colon_processed.csv...

================ BASELINE BENCHMARK RESULTS ================
Dataset Method          Classifier  Mean Accuracy  Std Accuracy  Mean Macro F1  Selected Features Count
  GOLUB   mRMR Logistic Regression         0.8333        0.0684         0.8246                      100
  GOLUB   mRMR    SVM (RBF Kernel)         0.8343        0.1036         0.8119                      100
  GOLUB   mRMR       Random Forest         0.8067        0.0991         0.7894                      100
  GOLUB   mRMR             XGBoost         0.8057        0.1292         0.8006                      100
  GOLUB  LASSO Logistic Regression         0.9171        0.0790   